# ⚙️ ShopBR — Feature Engineering

**Objetivo:** Transformar a série bruta em um conjunto de features que capturam:
- Padrões de autocorrelação (lags)
- Suavização e variância (rolling stats)
- Sazonalidade cíclica (sin/cos encoding)
- Flags de feriados comerciais
- Tendência linear

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, '../src')
from feature_engineering import criar_features

plt.style.use('dark_background')
print('Imports OK')

In [ ]:
df_raw = pd.read_csv('../data/raw/vendas.csv', parse_dates=['data'])
df_feat = criar_features(df_raw)
print(f'Shape: {df_feat.shape}')
print(f'Features: {df_feat.shape[1]} colunas')
df_feat.head()

## Lags e Autocorrelação
Verificamos a importância dos lags plotando a autocorrelação parcial (PACF):

In [ ]:
from statsmodels.graphics.tsaplots import plot_pacf
cat_df = df_raw[df_raw['categoria']=='Eletrônicos'].set_index('data')['unidades']
fig, ax = plt.subplots(figsize=(12,3))
plot_pacf(cat_df, lags=35, ax=ax, color='#00e5ff')
ax.set_title('PACF — Eletrônicos (lags até 35 dias)')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Distribuição das Features de Lag

In [ ]:
lag_cols = ['lag_1d','lag_7d','lag_14d','lag_28d']
corrs = df_feat[lag_cols + ['unidades']].corr()['unidades'][lag_cols]
fig, ax = plt.subplots(figsize=(6,3))
colors = ['#00e5ff' if c>0 else '#ff5e57' for c in corrs.values]
bars = ax.barh(corrs.index, corrs.values, color=colors, alpha=0.85)
ax.set_xlabel('Correlação com Target (unidades)')
ax.set_title('Correlação dos Lags com o Target')
ax.grid(True,alpha=0.3,axis='x')
plt.tight_layout(); plt.show()
print('Correlações:', corrs.round(3).to_dict())

## Sazonalidade Cíclica (sin/cos)

In [ ]:
theta = np.linspace(0, 2*np.pi, 12, endpoint=False)
fig, ax = plt.subplots(figsize=(5,5), subplot_kw={'projection':'polar'})
ax.scatter(theta, np.ones(12)*1.0, s=100, c=range(12), cmap='hsv', zorder=5)
meses = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
for i, (t, m) in enumerate(zip(theta, meses)):
    ax.text(t, 1.2, m, ha='center', va='center', fontsize=9, color='#e2eaf4')
ax.set_title('Encoding Cíclico do Mês', pad=20, color='#e2eaf4')
plt.tight_layout(); plt.show()

## Correlação de todas as Features com o Target

In [ ]:
df_clean = df_feat.dropna()
feature_cols = [c for c in df_clean.columns if c not in ['data','unidades','receita','preco_medio']]
corr_target = df_clean[feature_cols + ['unidades']].corr()['unidades'][feature_cols].abs().sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(8,6))
colors = [f'hsla({int(180 - i*8)},100%,60%,0.85)' for i in range(len(corr_target))]
colors = ['#00e5ff','#00d4ee','#00c3dd','#00b2cc','#00a1bb','#0090aa','#007f99','#006e88','#5d77',
          '#00e5cc','#00e5bb','#00e5aa','#a8ff3e','#97ff2e','#86ff1e','#75ff0e','#64ef00','#a29bfe','#9089fd','#7e77fc']
ax.barh(range(len(corr_target)), corr_target.values, color=colors[:len(corr_target)], alpha=0.9)
ax.set_yticks(range(len(corr_target)))
ax.set_yticklabels(corr_target.index, fontsize=9)
ax.set_xlabel('|Correlação| com Target')
ax.set_title('Top 20 Features por Correlação com Unidades Vendidas')
ax.grid(True,alpha=0.3,axis='x')
plt.tight_layout(); plt.show()